# 03. Machine Learning Modeling, Cross-Validation & Error Analysis
## Multi-Class Fault Classification Pipeline

In this notebook, we evaluate:
1. **Dummy Baseline** (Mathematical lower bound)
2. **Logistic Regression** (Linear benchmark)
3. **Random Forest Classifier** (Bagging ensemble)
4. **XGBoost Classifier** (Gradient boosted trees)

We use a strictly leak-free **70% Train, 15% Validation, 15% Test** split, with 5-Fold Stratified Cross-Validation on the training split.


In [ ]:
import sys, os
sys.path.append(os.path.abspath("../src"))

import pandas as pd
import numpy as np
import joblib

# Load evaluation results generated by src/evaluate.py
comparison_df = pd.read_csv("../reports/model_comparison_table.csv")
per_class_df = pd.read_csv("../reports/per_class_metrics.csv")
confused_pairs = pd.read_csv("../reports/confused_fault_pairs.csv")

print("=== FINAL MODEL BENCHMARK COMPARISON ON TEST SET (N=4,500) ===")
comparison_df


### 1. Analysis of Model Results
- **Dummy Classifier**: Yields 16.67% accuracy and 0.0476 Macro F1 (exact $1/6$ theoretical probability).
- **Logistic Regression**: High linear separability (98.07% Macro F1), but experiences slight boundary overlap.
- **Random Forest**: 99.02% Macro F1 with robust ensemble averaging.
- **XGBoost**: State-of-the-art performance at **99.20% Macro F1** and **99.20% Accuracy**.


In [ ]:
# Per-class performance of champion XGBoost model
xgb_classes = per_class_df[per_class_df['Model'] == 'XGBoost']
xgb_classes


### 2. In-Depth Error Analysis
Out of 4,500 held-out test samples, the champion XGBoost model committed only **36 misclassifications (0.80% error rate)**.
Let us examine the confused fault pairs.


In [ ]:
confused_pairs

### Physical Rationale for Misclassifications
1. **Reliability vs. Thermal Fault (27 / 36 errors)**: Both conditions exhibit extreme junction temperatures ($>95^\circ\text{C}$), high electromigration stress, and accelerated degradation. When an aging device operates near the thermal threshold, boundary ambiguity arises.
2. **Power vs. Thermal Fault (8 / 36 errors)**: High dynamic power dissipation ($P = V \cdot I$) directly heats the die ($T_j = T_a + P \cdot \theta_{ja}$). When heat sink cooling efficiency fluctuates, excessive power dissipation causes thermal symptoms.
3. **Zero Errors for Timing & Signal Integrity**: Undervoltage and negative SI margins provide clean orthogonal signatures.


### 3. Secondary Task: Mitigation Action Prediction
Predicting the 16 mitigation actions directly from raw electrical measurements (without knowing severity):


In [ ]:
import json
with open("../reports/mitigation_metrics.json", "r") as f:
    mit_metrics = json.load(f)
pd.DataFrame(mit_metrics).T
